In [ ]:
#1.Download dataset secara langsung melalui URL. 
#2.Ambil dataset tersebut secara langsung melalui URL menggunakan Python
 
import kagglehub
import pandas as pd
import os

# 1. Tentukan folder tujuan di laptop
target_folder = "data_hasil_download"
if not os.path.exists(target_folder):
    os.makedirs(target_folder)

# 2. Download dataset
path = kagglehub.dataset_download("himelsarder/retail-product-dataset-with-missing-values")

# 3. OTOMATIS cari file CSV di dalam folder hasil download
files = [f for f in os.listdir(path) if f.endswith('.csv')]

if len(files) > 0:
    file_name = files[0] # Ambil file CSV pertama yang ditemukan
    source_path = os.path.join(path, file_name)
    
    # 4. Baca dengan Pandas
    df = pd.read_csv(source_path)

    # 5. Simpan ke folder laptop
    final_destination = os.path.join(target_folder, "data_product_missing.csv")
    df.to_csv(final_destination, index=False)

    print(f"Berhasil! File '{file_name}' telah disimpan di: {final_destination}")
    display(df.head())
else:
    print("Waduh, tidak ada file CSV di folder tersebut. Cek isi folder:")
    print(os.listdir(path))

In [ ]:
#3.Eksplorasi data awal (Exploratory Data Analysis - EDA)

import pandas as pd

# 1. Memuat dataset
df = pd.read_csv('data_hasil_download/data_product_missing.csv')

# 2. Memeriksa struktur dasar data
print("--- Informasi Struktur Data ---")
print(df.info())

print("\n--- 5 Baris Pertama Data ---")
print(df.head())

# 3. Memeriksa kualitas data: Nilai yang hilang (Missing Values)
print("\n--- Jumlah Nilai yang Kosong per Kolom ---")
missing_data = df.isnull().sum()
persen_missing_data = (missing_data / len(df)) * 100
print(missing_data)

# Menampilkan persentase missing values
print("\n--- Presentase Nilai yang Kosong per Kolom ---")
print(persen_missing_data)

# 4. Memeriksa kualitas data: Baris duplikat
print("\n--- Jumlah Baris Duplikat ---")
print(df.duplicated().sum())

# 5. Ringkasan statistik untuk melihat distribusi data
print("\n--- Ringkasan Statistik ---")
print(df.describe(include='all'))

In [ ]:
#4.Lakukan preprocessing data:Penanganan missing values,Normalisasi data,
# Deteksi dan analisis outlier

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. Memuat dataset
df = pd.read_csv('data_hasil_download/data_product_missing.csv')

# --- a) Penanganan Missing Values ---
# Mengisi data kategorikal dengan nilai 'Unknown' atau Modus
df['Category'] = df['Category'].fillna('Unknown')
df['Stock'] = df['Stock'].fillna(df['Stock'].mode()[0])

# Mengisi data numerik dengan Median (lebih aman terhadap outlier dibanding Mean)
df['Price'] = df['Price'].fillna(df['Price'].median())
df['Rating'] = df['Rating'].fillna(df['Rating'].median())
df['Discount'] = df['Discount'].fillna(df['Discount'].median())

# --- b) Deteksi dan Analisis Outlier (Metode IQR) ---
def analyze_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    print(f"Fitur {column}: Ditemukan {len(outliers)} outlier.")
    return outliers

# Contoh analisis pada kolom Price dan Rating
outliers_price = analyze_outliers(df, 'Price')
outliers_rating = analyze_outliers(df, 'Rating')

# --- c) Normalisasi Data (Scaling ke rentang 0 - 1) ---
scaler = MinMaxScaler()
numerical_cols = ['Price', 'Rating', 'Discount']
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

#tranformasi log

# Menampilkan hasil akhir
print("\n--- Data Setelah Preprocessing (5 Baris Pertama) ---")
print(df.head())

# Menyimpan hasil ke file baru
df.to_csv('data_hasil_perbaikan/data_preprocessed.csv', index=False)

In [ ]:
#5. Uji normalitas data dan interpretasikan hasilnya.
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Memuat data hasil preprocessing
df = pd.read_csv('data_hasil_perbaikan/data_preprocessed.csv')

# Pilih kolom yang akan diuji (misal: 'Price')
column = 'Price'
data = df[column]

# 2. Uji Statistik
# Shapiro-Wilk Test (Cocok untuk sampel < 5000)
shapiro_stat, shapiro_p = stats.shapiro(data)

# D'Agostino's K^2 Test (Sering digunakan untuk Big Data)
k2_stat, k2_p = stats.normaltest(data)

print(f"--- Hasil Uji Statistik: {column} ---")
print(f"Shapiro-Wilk: Statistik={shapiro_stat:.4f}, p-value={shapiro_p:.4f}")
print(f"D'Agostino's K^2: Statistik={k2_stat:.4f}, p-value={k2_p:.4f}")

# 3. Interpretasi Sederhana
alpha = 0.05
if shapiro_p > alpha:
    print("\nKesimpulan: Data berdistribusi Normal (Gagal menolak H0)")
else:
    print("\nKesimpulan: Data TIDAK berdistribusi Normal (Menolak H0)")

# 4. Visualisasi untuk Konfirmasi
plt.figure(figsize=(12, 5))

# Plot Histogram & KDE
plt.subplot(1, 2, 1)
sns.histplot(data, kde=True, color='blue')
plt.title(f'Histogram & KDE: {column}')

# Plot Q-Q Plot (Semakin merapat ke garis diagonal, semakin normal)
plt.subplot(1, 2, 2)
stats.probplot(data, dist="norm", plot=plt)
plt.title(f'Q-Q Plot: {column}')

plt.tight_layout()
plt.savefig('uji_normalitas.png')

In [ ]:
#6 Perbandingan data sebelum dan sesudah pemrosesan
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Memuat kedua dataset
df_before = pd.read_csv('data_hasil_download/data_product_missing.csv')
df_after = pd.read_csv('data_hasil_perbaikan/data_preprocessed.csv')

# 2. Perbandingan Missing Values
print("--- Perbandingan Jumlah Missing Values ---")
comparison_missing = pd.DataFrame({
    'Sebelum': df_before.isnull().sum(),
    'Sesudah': df_after.isnull().sum()
})
print(comparison_missing)

# 3. Perbandingan Statistik (Contoh kolom Price)
print("\n--- Deskripsi Statistik Kolom 'Price' ---")
print("SEBELUM:\n", df_before['Price'].describe())
print("\nSESUDAH (Normalisasi):\n", df_after['Price'].describe())

# 4. Visualisasi Perbandingan Distribusi
plt.figure(figsize=(12, 10))

# Visualisasi Price Sebelum
plt.subplot(2, 2, 1)
sns.histplot(df_before['Price'].dropna(), kde=True, color='red')
plt.title('Distribusi Price (Sebelum)')

# Visualisasi Price Sesudah
plt.subplot(2, 2, 2)
sns.histplot(df_after['Price'], kde=True, color='green')
plt.title('Distribusi Price (Sesudah - Normalisasi)')

# Visualisasi Rating Sebelum
plt.subplot(2, 2, 3)
sns.histplot(df_before['Rating'].dropna(), kde=True, color='orange')
plt.title('Distribusi Rating (Sebelum)')

# Visualisasi Rating Sesudah
plt.subplot(2, 2, 4)
sns.histplot(df_after['Rating'], kde=True, color='blue')
plt.title('Distribusi Rating (Sesudah - Imputasi & Normalisasi)')

plt.tight_layout()
plt.savefig('perbandingan_preprocessing.png')
plt.show()